In [ ]:
# 0. DATA OVERVIEW — data_wip_v5.csv
# Objectifs :
# - Charger le master dataset (CSV séparé par ';')
# - Contrôles rapides : types, NaN, doublons, catégories
# - Générer un dictionnaire de données (docs/data_dictionary.csv)
# - Résumé exécutable pour le README

from pathlib import Path
import pandas as pd

# --- Chemins ---
ROOT = Path("..").resolve()
DATA = ROOT / "data"
DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA / "data_wip_v5.csv"
assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH}"
CSV_PATH

In [ ]:
# --- Chargement robuste (CSV FR ; séparateur ';') ---
# On force sep=';' (exports FR/Excel), encodage UTF-8 par défaut.
# Si UTF-8 échoue, on tente latin-1 (rarement utile si sep est correct).
try:
    df = pd.read_csv(CSV_PATH, sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, sep=",", encoding="latin-1")

print(f"Loaded {CSV_PATH.name} -> shape={df.shape}")
display(df.head(3))
display(df.info())

In [ ]:
# --- Contrôles rapides ---
dup_count = int(df.duplicated().sum())
na_counts = df.isna().sum().sort_values(ascending=False)

print("🔎 Lignes dupliquées :", dup_count)
display(na_counts.head(30))

In [ ]:
# --- Typage (aperçu) ---
dtypes_df = (
    pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)})
    .sort_values("dtype")
    .reset_index(drop=True)
)
dtypes_df.head(20)

In [ ]:
# --- Aperçu des catégories courtes (<= 15 valeurs uniques) ---
cat_preview = []
for c in df.columns:
    if df[c].dtype == "object":
        nunq = df[c].nunique(dropna=True)
        if nunq <= 15:
            vals = sorted([str(v) for v in df[c].dropna().unique().tolist()])[:15]
            cat_preview.append({"column": c, "n_unique": nunq, "values": vals})
pd.DataFrame(cat_preview)

In [ ]:
# --- Normalisations légères optionnelles ---
# (Décommente/ajuste si utile pour ton dataset)
# - convertir d’éventuels bytes -> str
def to_text(x):
    if isinstance(x, (bytes, bytearray)):
        try:
            return x.decode("utf-8", "ignore")
        except Exception:
            return str(x)
    return x


obj_cols = [c for c in df.columns if df[c].dtype == "object"]
for c in obj_cols:
    df[c] = df[c].map(to_text)

# Exemple spécifique "Code_Dpt" -> texte zfill(2)
if "Code_Dpt" in df.columns:
    df["Code_Dpt"] = (
        df["Code_Dpt"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(2)
    )

# Exemple dates (adapte la liste)
for cand in ["date", "Date", "DATE"]:
    if cand in df.columns:
        df[cand] = pd.to_datetime(df[cand], errors="coerce")

In [ ]:
# --- Dictionnaire de données (auto) ---
data_dict = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum().values,
        "example": [
            df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns
        ],
        "description": ["TODO: décrire la sémantique de la colonne"] * len(df.columns),
    }
)
data_dict_path = DOCS / "data_dictionary.csv"
data_dict.to_csv(data_dict_path, index=False, encoding="utf-8")
data_dict_path

In [ ]:
# --- Checks de cohérence simples (à adapter à tes colonnes) ---
issues = []

# Variables "standards" si présentes
COL_YEAR = "annee"
COL_REGION = "region"
COL_TYPE = "typologie"
COL_Y = "tonnage"

# Année plausible
if COL_YEAR in df.columns:
    bad_years = df.loc[~df[COL_YEAR].between(2000, 2100, inclusive="both"), COL_YEAR]
    if not bad_years.empty:
        issues.append(
            f"Valeurs 'annee' hors plage : {sorted(bad_years.dropna().unique().tolist())}"
        )

# Tonnage non négatif
if COL_Y in df.columns and pd.api.types.is_numeric_dtype(df[COL_Y]):
    if (df[COL_Y] < 0).any():
        issues.append("Des valeurs négatives détectées dans 'tonnage'.")

# Région/Typologie non vides (si colonnes présentes)
for col in [COL_REGION, COL_TYPE]:
    if col in df.columns and df[col].isna().all():
        issues.append(f"Toutes les valeurs de '{col}' sont NaN.")

if issues:
    print("⚠️ Problèmes détectés :")
    for e in issues:
        print(" -", e)
else:
    print("Checks de base OK.")

In [ ]:
# --- Résumé exécutable (à copier dans le README si besoin) ---
summary = {
    "file": str(CSV_PATH.relative_to(ROOT)),
    "shape": list(df.shape),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "n_duplicates": dup_count,
    "n_cols_with_missing": int((df.isna().sum() > 0).sum()),
    "data_dictionary": str(data_dict_path.relative_to(ROOT)),
    "columns_sample": df.columns[:10].tolist(),
}
summary